In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('/content/diabetes_data2.csv')

In [3]:
print(df.shape)

(768, 9)


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


In [5]:
# Replacing 0 as null value
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols] = df[cols].replace(0, np.nan)

In [6]:
# Imputing Null Value with KNNImputer
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)
df[cols] = knn_imputer.fit_transform(df[cols])

In [10]:
# FEATURE ENGINEERING
# ============================================================

# --- 1. Interaction Features (domain-meaningful products) ---
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Glucose_Insulin'] = df['Glucose'] * df['Insulin']
df['BMI_Age'] = df['BMI'] * df['Age']
df['Insulin_SkinThickness'] = df['Insulin'] * df['SkinThickness']

# --- 2. Ratio Features ---
df['Glucose_Insulin_Ratio'] = df['Glucose'] / (df['Insulin'] + 1e-6)
df['BMI_Age_Ratio'] = df['BMI'] / (df['Age'] + 1e-6)

# --- 3. Polynomial Features (squared terms) ---
df['Glucose_sq'] = df['Glucose'] ** 2
df['BMI_sq'] = df['BMI'] ** 2
df['Age_sq'] = df['Age'] ** 2

# --- 4. Age Binning (clinical age groups) ---
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[0, 30, 45, 60, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)
df['Age_Group'] = df['Age_Group'].astype(str)

# One-hot encode age group
df = pd.get_dummies(df, columns=['Age_Group'], drop_first=True)

# --- 5. BMI Category (WHO standard) ---
df['BMI_Category'] = pd.cut(
    df['BMI'],
    bins=[0, 18.5, 24.9, 29.9, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)
df['BMI_Category'] = df['BMI_Category'].astype(str)
df = pd.get_dummies(df, columns=['BMI_Category'], drop_first=True)

# --- 6. Glucose Category (clinical thresholds) ---
df['Glucose_Level'] = pd.cut(
    df['Glucose'],
    bins=[0, 99, 125, 300],
    labels=['Normal', 'Prediabetes', 'Diabetes']
)
df['Glucose_Level'] = df['Glucose_Level'].astype(str)
df = pd.get_dummies(df, columns=['Glucose_Level'], drop_first=True)

# --- 7. Insulin Sensitivity Proxy ---
# HOMA-IR approximation (proxy for insulin resistance)
df['HOMA_IR'] = (df['Glucose'] * df['Insulin']) / 405

# --- 8. Pregnancies Risk Flag ---
df['High_Pregnancies'] = (df['Pregnancies'] > 3).astype(int)

# --- 9. Log Transforms (reduce skewness on right-skewed features) ---
df['Log_Insulin'] = np.log1p(df['Insulin'])
df['Log_DiabetesPedigreeFunction'] = np.log1p(df['DiabetesPedigreeFunction'])

# --- 10. Composite Risk Score (custom domain knowledge) ---
df['Risk_Score'] = (
    (df['Glucose'] / 100) +
    (df['BMI'] / 25) +
    (df['Age'] / 50) +
    (df['DiabetesPedigreeFunction'])
)

In [12]:
print(df.columns)

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome', 'Glucose_BMI',
       'Glucose_Insulin', 'BMI_Age', 'Insulin_SkinThickness',
       'Glucose_Insulin_Ratio', 'BMI_Age_Ratio', 'Glucose_sq', 'BMI_sq',
       'Age_sq', 'Age_Group_Middle', 'Age_Group_Senior', 'Age_Group_Young',
       'BMI_Category_Obese', 'BMI_Category_Overweight',
       'BMI_Category_Underweight', 'Glucose_Level_Normal',
       'Glucose_Level_Prediabetes', 'HOMA_IR', 'High_Pregnancies',
       'Log_Insulin', 'Log_DiabetesPedigreeFunction', 'Risk_Score'],
      dtype='object')


In [13]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Pregnancies                   768 non-null    float64
 1   Glucose                       768 non-null    float64
 2   BloodPressure                 768 non-null    float64
 3   SkinThickness                 768 non-null    float64
 4   Insulin                       768 non-null    float64
 5   BMI                           768 non-null    float64
 6   DiabetesPedigreeFunction      768 non-null    float64
 7   Age                           768 non-null    float64
 8   Outcome                       768 non-null    int64  
 9   Glucose_BMI                   768 non-null    float64
 10  Glucose_Insulin               768 non-null    float64
 11  BMI_Age                       768 non-null    float64
 12  Insulin_SkinThickness         768 non-null    float64
 13  Gluco

In [ ]:
# Cap outliers using IQR
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    return df

for col in df.columns[:-1]:  # exclude target
    df = cap_outliers(df, col)